# TravelMind Notebook 1: Foundations & Workflow Patterns
### IBS Agentic AI Practitioner Bootcamp | Day 6 | Advanced Strands

**Goal:** turn the workflow patterns into runnable Strands code on one evolving airline-support agent, and measure the **three dials (cost, quality signal, latency)** for every version.

**You will build:**
- **v1**: the augmented single agent (recap) + a mock airline operations layer
- A **measurement harness** that records latency, tokens, and cost per run
- **v2** Prompt Chaining, **v3** Routing, **v4** Parallelization (sectioning + voting)
- A **dashboard** comparing every version on the three dials

**Mental model to hold the whole time:** *who controls the path?* In this notebook, **you** do. These are workflows: fixed, cheap, predictable. Notebook 2 hands the path to the model.

## How to run

**VS Code**
1. Open this `.ipynb`, select a Python 3.10+ kernel (a virtual environment is fine).
2. Ensure AWS credentials are available to boto3 (an SSO/role session, or `aws configure`), with Bedrock model access enabled for Claude Haiku 4.5 in `us-east-1`.
3. Run cells top to bottom. The first cell installs dependencies.

**Google Colab**
1. Upload the notebook, then run the install cell.
2. Provide credentials for the session: `import os; os.environ["AWS_ACCESS_KEY_ID"]=...` etc. (temporary role credentials preferred), and `os.environ["AWS_REGION"]="us-east-1"`.
3. Run top to bottom.

**Cost note:** these cells make real Bedrock calls on Haiku (cheap). The mock airline backend is local, so only model tokens cost money.

In [ ]:
%pip install -q strands-agents strands-agents-tools nest_asyncio matplotlib

## Environment configuration

We tier models on purpose: **Haiku by default**. The `us.` prefix on the model id selects a **cross-region inference profile**, which these models require on Bedrock. Getting this prefix wrong is the single most common first-day error.

In [ ]:
import os, time, json, asyncio, contextlib
import nest_asyncio
nest_asyncio.apply()  # allow asyncio.run(...) inside a notebook cell

from strands import Agent, tool
from strands.models import BedrockModel

REGION = os.environ.get("AWS_REGION", "us-east-1")

# Cheapest tier by default. Promote a stage to Sonnet only when a failure forces it.
HAIKU_ID  = "us.anthropic.claude-haiku-4-5-20251001-v1:0"   # us. => cross-region inference profile
SONNET_ID = "us.anthropic.claude-sonnet-4-5-20250929-v1:0"  # promote target (needs its own model access)

haiku = BedrockModel(model_id=HAIKU_ID, region_name=REGION, temperature=0.3)
# sonnet = BedrockModel(model_id=SONNET_ID, region_name=REGION, temperature=0.3)  # enable when needed

print("Region:", REGION)
print("Default model:", HAIKU_ID)

### First-day errors and their fixes

| Error | Cause | Fix |
|---|---|---|
| `on-demand throughput isn't supported` | Missing region prefix | Use the `us.` inference-profile id (as above) |
| `ValidationException: model identifier is invalid` | Region has no inference profile for that id | Use a base model id, or switch region |
| `ThrottlingException` | Too many requests | Add retries via `boto_client_config` (shown in production notes) |
| `AccessDeniedException` | Model access not enabled / IAM gap | Enable model access in Bedrock console; grant `bedrock:InvokeModel` |

## The TravelMind operations layer (mocked)

Realistic airline objects, deterministic fake data, so the notebook runs with no reservation system attached. Each tool is a normal Python function with a docstring; Strands turns it into a callable tool for the agent.

**Anchor booking:** PNR `JX48Q2`, passenger surname `Rao`, Gold tier, first segment `BLR to DEL` **cancelled by the airline**.

In [ ]:
# --- Mock airline data ---
_PNRS = {
    "JX48Q2": {
        "surname": "Rao",
        "passenger_id": "P-100294",
        "loyalty_tier": "Gold",
        "segments": [
            {"flight": "6E-317", "from": "BLR", "to": "DEL", "dep": "2026-07-05T08:10",
             "status": "CANCELLED", "fare_basis": "TA21R"},
            {"flight": "6E-512", "from": "DEL", "to": "BOM", "dep": "2026-07-05T13:40",
             "status": "ON TIME", "fare_basis": "TA21R"},
        ],
        "ancillaries": {"seat": "14C (paid)", "bags": 1},
    }
}
_FARE_RULES = {
    "TA21R": {"refundable": False, "change_fee": 3000, "currency": "INR",
              "notes": "Non-refundable on voluntary change. Airline-caused (involuntary) changes waive fees."},
}


@tool
def get_pnr(record_locator: str, surname: str) -> str:
    """Look up a booking by record locator and surname (identity check).

    Args:
        record_locator: 6-character PNR, e.g. 'JX48Q2'.
        surname: Passenger surname for verification.
    Returns:
        JSON string of the booking, or an error if not found / surname mismatch.
    """
    rec = _PNRS.get(record_locator.upper())
    if not rec or rec["surname"].lower() != surname.lower():
        return json.dumps({"error": "PNR not found or surname mismatch"})
    return json.dumps(rec)


@tool
def get_fare_rules(fare_basis: str) -> str:
    """Return fare rules (refundability, change fee) for a fare basis code.

    Args:
        fare_basis: Fare basis code, e.g. 'TA21R'.
    Returns:
        JSON string of fare rules.
    """
    return json.dumps(_FARE_RULES.get(fare_basis.upper(), {"error": "unknown fare basis"}))


@tool
def search_reaccommodation(origin: str, destination: str, after: str) -> str:
    """Find alternative flights to re-accommodate a passenger after a disruption.

    Args:
        origin: Origin airport code, e.g. 'BLR'.
        destination: Destination airport code, e.g. 'BOM'.
        after: ISO datetime; only flights departing after this are returned.
    Returns:
        JSON string list of candidate flights.
    """
    options = [
        {"flight": "6E-333", "from": origin, "to": destination, "dep": "2026-07-05T11:20", "seats": 4},
        {"flight": "AI-809", "from": origin, "to": destination, "dep": "2026-07-05T15:05", "seats": 9},
    ]
    return json.dumps(options)


@tool
def get_loyalty(passenger_id: str) -> str:
    """Return loyalty tier and benefits for a passenger.

    Args:
        passenger_id: Internal passenger id, e.g. 'P-100294'.
    Returns:
        JSON string of tier and benefits.
    """
    benefits = {"Gold": ["priority rebooking", "waived change fee on same-day", "lounge access"]}
    return json.dumps({"passenger_id": passenger_id, "tier": "Gold", "benefits": benefits["Gold"]})


@tool
def check_refund_eligibility(record_locator: str, reason: str) -> str:
    """Assess refund eligibility from the booking's fare rules and the stated reason.

    Args:
        record_locator: PNR.
        reason: Free-text reason, e.g. 'flight cancelled by airline'.
    Returns:
        JSON string with an eligibility signal and the controlling rule.
    """
    rec = _PNRS.get(record_locator.upper(), {})
    if not rec:
        return json.dumps({"error": "PNR not found"})
    fb = rec["segments"][0]["fare_basis"]
    rules = _FARE_RULES.get(fb, {})
    airline_caused = any(k in reason.lower() for k in ["cancel", "delay", "airline", "irrops"])
    eligible = airline_caused or rules.get("refundable", False)
    return json.dumps({"eligible": eligible, "airline_caused": airline_caused, "rule": rules.get("notes", "")})

print("Tools ready:", [t for t in ["get_pnr","get_fare_rules","search_reaccommodation","get_loyalty","check_refund_eligibility"]])

> **What changes in production**
> - These functions become real calls to the reservation system, fare engine, and loyalty database (least-privilege service credentials, not keys in code).
> - Identity verification is a real auth step, not a string compare.
> - Every tool call is traced and rate-limited; failures return typed errors, not silent empties.

## The measurement harness

One helper wraps every run so we can compare versions honestly. `meter(label)` times a block and sums the token cost of every model call inside it; `metered(agent, prompt)` runs an agent and records its usage.

**Pricing note:** the numbers below are illustrative Bedrock list prices (USD per 1M tokens). Confirm current pricing for the client's region before quoting. The *formula* is the durable skill; swap in your negotiated rates.

In [ ]:
# Illustrative Bedrock on-demand list prices, USD per 1M tokens: (input, output).
# VERIFY against current Bedrock pricing for your region before quoting.
PRICES = {"haiku": (1.00, 5.00), "sonnet": (3.00, 15.00)}

_LEDGER = []   # (tier, {"input":..,"output":..}) appended per model call inside a metered block
RESULTS = {}   # label -> {latency_s, input_tokens, output_tokens, calls, cost_usd}


def usage_of(result) -> dict:
    """Pull token usage from a Strands result (single-agent or multi-agent)."""
    u = getattr(getattr(result, "metrics", None), "accumulated_usage", None)
    if u is None:
        u = getattr(result, "accumulated_usage", None) or {}
    def g(*keys):
        for k in keys:
            if k in u:
                return u[k]
        return 0
    return {"input": g("inputTokens", "input_tokens"),
            "output": g("outputTokens", "output_tokens")}


def metered(agent, prompt, tier="haiku"):
    """Run an agent once, record its usage on the active ledger, return the text."""
    res = agent(prompt)
    _LEDGER.append((tier, usage_of(res)))
    return str(res)


@contextlib.contextmanager
def meter(label):
    """Time a block; sum cost across every metered() call inside it; store the dials."""
    _LEDGER.clear()
    t0 = time.time()
    try:
        yield
    finally:
        latency = time.time() - t0
        cost = tin = tout = 0
        for tier, u in _LEDGER:
            pin, pout = PRICES[tier]
            cost += u["input"] / 1e6 * pin + u["output"] / 1e6 * pout
            tin += u["input"]; tout += u["output"]
        RESULTS[label] = {"latency_s": round(latency, 2), "input_tokens": tin,
                          "output_tokens": tout, "calls": len(_LEDGER),
                          "cost_usd": round(cost, 6)}
        r = RESULTS[label]
        print(f"[{label}]  calls={r['calls']}  latency={r['latency_s']}s  "
              f"tokens={tin}in+{tout}out  cost=${r['cost_usd']:.6f}")

---
## Section A: v1: the augmented single agent (recap)

One agent, a couple of tools, the agent loop decides when to call them. This is yesterday, restated as our baseline.

```mermaid
flowchart LR
    U([Customer]) --> A[TravelMind v1]
    A -->|call| T1[get_pnr]
    A -->|call| T2[search_reaccommodation]
    T1 --> A
    T2 --> A
    A --> R([Answer])
```

In [ ]:
TRAVELMIND_V1_PROMPT = """You are TravelMind, an airline customer support agent.
Verify identity with record locator + surname before sharing booking details.
Use tools to look up facts. Never invent flight numbers, fares, or policies.
Be concise and warm."""

travelmind_v1 = Agent(model=haiku, name="travelmind_v1",
                      system_prompt=TRAVELMIND_V1_PROMPT,
                      tools=[get_pnr, search_reaccommodation])

q_simple = "Hi, this is Rao. What's the status of my booking JX48Q2?"

with meter("v1_single_agent"):
    answer = metered(travelmind_v1, q_simple)
    print(answer)

> **What changes in production**
> - Model credentials come from an IAM role, never hardcoded; region comes from environment.
> - Add `boto_client_config` retries for throttling, and turn on OpenTelemetry traces so every tool call and token count is captured.
> - Apply a Bedrock guardrail on the customer-facing output.

---
## Section B: v2: Prompt Chaining

Decompose a fixed task into ordered steps: extract intent, resolve facts with tools, then write the reply. **You** wrote the order.

```mermaid
flowchart LR
    In([Request]) --> S1[extractor: intent + entities]
    S1 --> S2[resolver: gather facts via tools]
    S2 --> S3[writer: customer reply]
    S3 --> Out([Reply])
```

Sequence of a single change request through the chain:

```mermaid
sequenceDiagram
    participant C as Customer
    participant E as Extractor
    participant R as Resolver
    participant W as Writer
    C->>E: change request (raw text)
    E->>R: intent + entities (JSON)
    R->>R: get_pnr / get_fare_rules / search_reaccommodation
    R->>W: options + facts
    W->>C: warm, correct reply
```

In [ ]:
extractor = Agent(model=haiku, name="extractor",
    system_prompt="Extract the customer's intent and entities (PNR, surname, dates) as compact JSON. Output JSON only.")

resolver = Agent(model=haiku, name="resolver",
    system_prompt="Given intent + entities, use tools to gather the facts needed to resolve a flight change. Summarize the options plainly.",
    tools=[get_pnr, get_fare_rules, search_reaccommodation])

writer = Agent(model=haiku, name="writer",
    system_prompt="Turn the resolver's findings into a warm, correct, concise customer reply.")

q_change = "This is Rao, PNR JX48Q2. My first flight got cancelled. What are my options to still reach BOM today?"

with meter("v2_prompt_chaining"):
    intent = metered(extractor, q_change)
    facts  = metered(resolver, "Intent+entities:\n" + intent + "\n\nOriginal message:\n" + q_change)
    reply  = metered(writer, facts)
    print(reply)

> **What changes in production**
> - Insert a programmatic gate between steps (for example, fail fast if `get_pnr` returns an error) so a bad PNR never reaches the writer.
> - Cache the three static system prompts (Bedrock prompt caching) since they never change across requests.

---
## Section C: v3: Routing

Classify first with a cheap model, then dispatch to one focused specialist. The heavy work only runs on the branch taken. This is the cheapest cost-and-quality win on the ladder.

```mermaid
flowchart TD
    In([Message]) --> R{Haiku classifier}
    R -->|status| A1[status_agent]
    R -->|change| A2[change_agent]
    R -->|refund| A3[refund_agent]
```

We also run a single "mega agent" that carries every tool on every call, to contrast token and cost profiles.

In [ ]:
CATEGORIES = ["status", "change", "refund", "baggage", "complaint"]

classifier = Agent(model=haiku, name="classifier",
    system_prompt="Classify the customer message into exactly one label from "
                  + str(CATEGORIES) + ". Output only the label, nothing else.")

status_agent = Agent(model=haiku, name="status_agent",
    system_prompt="Handle flight-status questions. Verify identity, then report each segment's status.",
    tools=[get_pnr])
change_agent = Agent(model=haiku, name="change_agent",
    system_prompt="Handle flight changes. Check fare rules and find re-accommodation.",
    tools=[get_pnr, get_fare_rules, search_reaccommodation])
refund_agent = Agent(model=haiku, name="refund_agent",
    system_prompt="Handle refunds. Always check eligibility against fare rules before any promise.",
    tools=[get_pnr, check_refund_eligibility])

SPECIALISTS = {"status": status_agent, "change": change_agent, "refund": refund_agent}

def route(msg):
    label = metered(classifier, msg).strip().lower().split()[0]
    label = label if label in SPECIALISTS else "status"
    return label, metered(SPECIALISTS[label], msg)

q_refund = "Rao here, JX48Q2. My flight was cancelled by the airline. Can I get a refund?"

with meter("v3_routing"):
    label, reply = route(q_refund)
    print("routed ->", label)
    print(reply)

In [ ]:
# Contrast: one agent holding every tool, answering the same query.
mega_agent = Agent(model=haiku, name="mega_agent",
    system_prompt=TRAVELMIND_V1_PROMPT + "\nYou also handle changes, refunds, baggage, and complaints.",
    tools=[get_pnr, get_fare_rules, search_reaccommodation, get_loyalty, check_refund_eligibility])

with meter("v3_baseline_mega_agent"):
    print(metered(mega_agent, q_refund))

**Read the two rows in the dashboard.** The router spends a tiny classify call, then runs a specialist that carries only the tools it needs. The mega agent ships every tool schema on every turn. On real traffic across thousands of queries, that difference compounds into a materially larger bill.

> **What changes in production**
> - Route the cheap classifier to the cheapest model, and consider promoting only the hardest branch (for example, complaints) to Sonnet.
> - Log the routing decision for every message so you can audit misroutes and retrain the classifier prompt.

---
## Section D: v4: Parallelization

**Sectioning:** fare-rules, re-accommodation, and loyalty checks are independent, so run them concurrently and merge. Wall-clock drops to the slowest one instead of the sum.

**Voting:** for a high-stakes refund decision, run the judgment several times and take the majority.

```mermaid
flowchart TD
    In([Change request]) --> C[coordinator]
    C --> W1[fare_agent]
    C --> W2[reaccom_agent]
    C --> W3[loyalty_agent]
    W1 --> AGG[aggregator]
    W2 --> AGG
    W3 --> AGG
    AGG --> Out([Complete reply])
```

We use `asyncio.to_thread` to run the three synchronous agent calls concurrently (Bedrock calls are I/O-bound, so threads give real overlap). `nest_asyncio` lets `asyncio.run` work inside the notebook.

In [ ]:
fare_agent = Agent(model=haiku, name="fare_agent",
    system_prompt="Report only the fare rules relevant to a change for this PNR.",
    tools=[get_pnr, get_fare_rules])
reaccom_agent = Agent(model=haiku, name="reaccom_agent",
    system_prompt="Report only re-accommodation options for this disruption.",
    tools=[get_pnr, search_reaccommodation])
loyalty_agent = Agent(model=haiku, name="loyalty_agent",
    system_prompt="Report only the loyalty tier and benefits relevant to a change.",
    tools=[get_pnr, get_loyalty])
aggregator = Agent(model=haiku, name="aggregator",
    system_prompt="Merge the fare, re-accommodation, and loyalty findings into one clear customer reply.")

async def gather_change(msg):
    fare, reaccom, loyalty = await asyncio.gather(
        asyncio.to_thread(metered, fare_agent, msg, "haiku"),
        asyncio.to_thread(metered, reaccom_agent, msg, "haiku"),
        asyncio.to_thread(metered, loyalty_agent, msg, "haiku"),
    )
    return metered(aggregator,
        "Fare:\n" + fare + "\n\nRe-accommodation:\n" + reaccom + "\n\nLoyalty:\n" + loyalty)

with meter("v4_parallel_sectioning"):
    print(asyncio.run(gather_change(q_change)))

In [ ]:
# Baseline: the same three checks run one after another. Same cost, higher latency.
def sequential_change(msg):
    fare    = metered(fare_agent, msg)
    reaccom = metered(reaccom_agent, msg)
    loyalty = metered(loyalty_agent, msg)
    return metered(aggregator,
        "Fare:\n" + fare + "\n\nRe-accommodation:\n" + reaccom + "\n\nLoyalty:\n" + loyalty)

with meter("v4_sequential_baseline"):
    print(sequential_change(q_change))

In [ ]:
# Voting: run the refund judgment N times, take the majority. Spends cost to buy confidence.
def refund_vote(msg, n=3):
    votes = []
    for _ in range(n):
        out = metered(refund_agent,
            msg + "\nAnswer strictly 'ELIGIBLE' or 'NOT ELIGIBLE' on the first line, then explain.")
        first = out.strip().upper().splitlines()[0]
        votes.append("ELIGIBLE" if ("ELIGIBLE" in first and "NOT" not in first) else "NOT ELIGIBLE")
    decision = max(set(votes), key=votes.count)
    return decision, votes

with meter("v4_voting_refund"):
    decision, votes = refund_vote(q_refund, n=3)
    print("votes:", votes, "-> decision:", decision)

> **What changes in production**
> - Cap concurrency (a bounded thread pool or semaphore) so a traffic spike does not fan out into a throttling storm.
> - Reserve voting for genuinely irreversible decisions; it multiplies cost by N. Log each vote for audit.
> - Set a per-branch timeout so one slow tool cannot stall the aggregate.

---
## Section E: the three dials, side by side

Run this after the sections above. It reads whatever is in `RESULTS` and plots latency, cost, and tokens per version. This is the habit to build: never choose a pattern without seeing its dials.

In [ ]:
import matplotlib.pyplot as plt

order = ["v1_single_agent", "v2_prompt_chaining", "v3_routing", "v3_baseline_mega_agent",
         "v4_parallel_sectioning", "v4_sequential_baseline", "v4_voting_refund"]
labels = [k for k in order if k in RESULTS]

lat  = [RESULTS[k]["latency_s"] for k in labels]
cost = [RESULTS[k]["cost_usd"] * 1000 for k in labels]  # scaled for visibility
toks = [RESULTS[k]["input_tokens"] + RESULTS[k]["output_tokens"] for k in labels]

fig, ax = plt.subplots(1, 3, figsize=(16, 4.5))
ax[0].barh(labels, lat);  ax[0].set_title("Latency (s) - lower better");   ax[0].invert_yaxis()
ax[1].barh(labels, cost); ax[1].set_title("Cost (USD x1000) - lower better"); ax[1].invert_yaxis()
ax[2].barh(labels, toks); ax[2].set_title("Total tokens - lower better");   ax[2].invert_yaxis()
plt.tight_layout(); plt.show()

print(f"\n{'version':<28}{'calls':>6}{'latency_s':>11}{'in_tok':>9}{'out_tok':>9}{'cost_usd':>12}")
for k in labels:
    r = RESULTS[k]
    print(f"{k:<28}{r['calls']:>6}{r['latency_s']:>11}{r['input_tokens']:>9}{r['output_tokens']:>9}{r['cost_usd']:>12.6f}")

### What the dials should teach you

- **v3 routing vs v3 mega agent:** routing usually wins on tokens and cost because the specialist carries fewer tool schemas.
- **v4 parallel vs v4 sequential:** near-identical cost and tokens, lower latency for parallel. Parallel buys *time*, not *money*.
- **v4 voting:** roughly 3x the calls of a single judgment. You are spending cost to buy confidence on an irreversible decision.

**Numbers vary per run** (model nondeterminism, network). The *relationships* hold, and those relationships are the intuition to internalize.

---
## Recap and what is next

You built four versions of TravelMind and measured all three dials for each:
- **v1** augmented agent, **v2** chaining, **v3** routing, **v4** parallelization.
- You own the path in every one. These are workflows: cheap, predictable, easy to debug.

**Notebook 2** hands the path to the model: **v5** orchestrator-workers (agents as tools) and **v6** evaluator-optimizer (a policy critic loop). Cost goes up; so does flexibility. Same harness, same TravelMind, so the dials stay comparable.